In [1]:
import torch
import CT.Models.misc as misc
import CT.Inference.Kolmogorov.performance as performance
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Emu_file_path = '/scratch/ql2221/thermalizer_data/wandb_data/wandb/run-20251028_103612-lmys64ex/files/checkpoint_best.p'
CT_file_path = '/scratch/ql2221/thermalizer_data/wandb_data/wandb/run-20251117_121401-tgz3mmhq/files/checkpoint_last.p'

CT = misc.load_diffusion_model(CT_file_path).to(device)
Emu = misc.load_model(Emu_file_path).to(device)

In [2]:
data_dict = torch.load("/scratch/ql2221/thermalizer_data/kolmogorov/reynold10k/Long_numerical_rollout.p")

In [3]:
data = data_dict["data"]
print(data.shape)
x = data[:,:,:,:]
print(x.shape)
x = x/4.44
del data, data_dict

torch.Size([5, 50000, 64, 64])
torch.Size([5, 50000, 64, 64])


In [4]:
emu_rollout = performance.run_emu(x[:,0:1],emu = Emu, n_steps=30000,silent=False,sigma=None)

100%|██████████| 29999/29999 [03:08<00:00, 159.46it/s]


In [5]:
short_lag = torch.tensor([1]).to(device)
long_lag = torch.tensor([120]).to(device)
Therm_rollout, _ = performance.run_conditional_emu(x[:,0:1], emu = Emu, therm=CT, n_steps=30000, short_lag = short_lag, long_lag = long_lag, s = 20, freq = 25, silence=True, sigma=None, Regression = True, device = device)

In [6]:
import matplotlib.pyplot as plt
from matplotlib import animation
def make_movie(state_vector, save_path="emulator_movie.mp4", fps=30, vmin=None, vmax=None, stride=10):
    """
    Create a movie from a 4D tensor of shape (batch, time, height, width), using only every `stride`-th frame.
    
    Parameters:
    - state_vector: torch.Tensor of shape (B, T, H, W)
    - save_path: output video file path
    - fps: frames per second for the movie
    - vmin, vmax: color scale
    - stride: only include every `stride`th frame in the video
    """
    B, T, H, W = state_vector.shape
    data = state_vector[0].cpu().numpy()

    fig, ax = plt.subplots()
    im = ax.imshow(data[0], cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title("Timestep 0")
    fig.colorbar(im, ax=ax)

    # Frame indices to use
    frame_indices = list(range(0, T, stride))

    def update(frame_idx):
        frame = frame_indices[frame_idx]
        im.set_data(data[frame])
        ax.set_title(f"Timestep {frame}")
        return im,

    ani = animation.FuncAnimation(fig, update, frames=len(frame_indices), blit=False, repeat=False)
    ani.save(save_path, fps=fps)
    plt.close(fig)
    print("movie saved")

In [7]:
make_movie(Therm_rollout, save_path="emu_rollout.mp4", fps=30)

movie saved


In [16]:
x = x.to(device)
empty = torch.zeros(4,emu_rollout.shape[1],emu_rollout.shape[2],emu_rollout.shape[3]).to(device)
tensor = torch.cat((x[:,:emu_rollout.shape[1]], emu_rollout, Therm_rollout), dim = 0)
titles = [ "GT", "non", "non", "non", "non", "Emu", "non", "non", "non", "non", "Therm", "non", "non", "non", "non"]

In [17]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm import tqdm

def make_movie(
    state_vector,
    save_path="emulator_movie.mp4",
    fps=30,
    vmin=None,
    vmax=None,
    stride=10,
    titles=None,
    cmap="viridis",
    interpolation="nearest",
    dpi=150,
    bitrate=1800,
):
    """
    Create a 3x5 grid movie from a 4D tensor/array of shape (B, T, H, W), using only every `stride`-th frame.
    - B must be 15 (3 rows x 5 columns).
    - One shared colorbar per row (fixed over time).
    - Optional custom title per panel via `titles` (len 15, row-major).
    - tqdm progress bar while writing with ffmpeg.

    Parameters
    ----------
    state_vector : torch.Tensor | np.ndarray of shape (B, T, H, W)
    save_path    : str, output video file path (e.g., "emulator_movie.mp4")
    fps          : int, frames per second
    vmin, vmax   : float or None
        If provided, used as a global color scale across ALL rows.
        If None, each row gets auto-computed (fixed) limits over INCLUDED frames.
    stride       : int, include every `stride`-th frame
    titles       : list[str] or None, length 15 (row-major: r0 c0..4, r1 c0..4, r2 c0..4)
    cmap         : str, matplotlib colormap name
    interpolation: str, imshow interpolation (e.g., "nearest", "bilinear", "bicubic")
    dpi          : int, figure DPI for saved video
    bitrate      : int, bitrate for FFMpegWriter
    """

    # Convert to numpy without requiring torch as a dependency at import time
    if hasattr(state_vector, "detach"):  # likely a torch.Tensor
        arr = state_vector.detach().cpu().numpy()
    else:
        arr = np.asarray(state_vector)

    if arr.ndim != 4:
        raise ValueError(f"Expected [B, T, H, W], got {arr.shape}")
    B, T, H, W = arr.shape
    if B != 15:
        raise ValueError(f"B must be 15 for a 3x5 grid, got B={B}")
    if T < 1:
        raise ValueError("T must be >= 1")

    # Frame indices to use
    stride = max(1, int(stride))
    frame_indices = list(range(0, T, stride))
    if frame_indices[-1] != T - 1:
        frame_indices.append(T - 1)  # ensure last frame is included

    # Helper for row/col -> batch index
    def bidx(r, c):
        return r * 5 + c

    # Determine color limits
    row_limits = []
    if vmin is not None or vmax is not None:
        # Global scale across all rows
        gmin, gmax = vmin, vmax
        if gmin is None or gmax is None:
            used = arr[:, frame_indices, :, :]
            amin = float(np.nanmin(used))
            amax = float(np.nanmax(used))
            if gmin is None:
                gmin = amin
            if gmax is None:
                gmax = amax
        row_limits = [(gmin, gmax)] * 3
    else:
        # Per-row fixed scale computed from included frames only
        for r in range(3):
            row_data = arr[bidx(r, 0):bidx(r, 0)+5, :, :, :]  # [5, T, H, W]
            row_used = row_data[:, frame_indices, :, :]
            rmin = float(np.nanmin(row_used))
            rmax = float(np.nanmax(row_used))
            row_limits.append((rmin, rmax))

    # Titles
    if titles is None:
        titles = [f"Panel {i+1}" for i in range(15)]
    if len(titles) != 15:
        raise ValueError("`titles` must have length 15 (row-major order).")

    # Figure and axes
    fig, axes = plt.subplots(3, 5, figsize=(15, 9), dpi=dpi, squeeze=False)

    # Create initial images
    ims = []  # keep references for animation update
    for r in range(3):
        vmin_r, vmax_r = row_limits[r]
        for c in range(5):
            ax = axes[r, c]
            im = ax.imshow(
                arr[bidx(r, c), frame_indices[0]],
                cmap=cmap,
                vmin=vmin_r,
                vmax=vmax_r,
                interpolation=interpolation,
                origin="upper",
                animated=True,
            )
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(titles[bidx(r, c)], fontsize=10)
            ims.append(im)

        # One colorbar per row, shared across the five axes in that row
        cbar = fig.colorbar(
            ims[r*5 + 4],  # any image from this row
            ax=list(axes[r, :]),
            orientation="vertical",
            fraction=0.046,
            pad=0.02
        )
        cbar.set_label(
            f"Row {r+1} scale [{row_limits[r][0]:.3g}, {row_limits[r][1]:.3g}]",
            fontsize=9
        )

    # Global suptitle shows the current frame index
    suptitle = fig.suptitle(f"Timestep {frame_indices[0]}", fontsize=12)

    # Update function used by both preview animation and manual writer loop
    def update(i):
        f = frame_indices[i]
        for r in range(3):
            for c in range(5):
                ims[r*5 + c].set_data(arr[bidx(r, c), f])
        suptitle.set_text(f"Timestep {f}")
        return ims

    # Build the animation object (useful if you want to preview in notebooks),
    # but we'll save manually to expose tqdm progress.
    _ = animation.FuncAnimation(
        fig, update, frames=len(frame_indices), blit=False, repeat=False
    )

    # Save with a manual loop + tqdm progress bar
    writer = animation.FFMpegWriter(fps=fps, bitrate=bitrate)
    with writer.saving(fig, save_path, dpi=dpi):
        for i in tqdm(range(len(frame_indices)), desc="Rendering movie"):
            update(i)
            writer.grab_frame()

    plt.close(fig)
    print(f"🎬 Movie saved: {save_path}")


In [18]:
make_movie(
    state_vector = tensor,
    save_path="compare_movie.mp4",
    fps=30,
    vmin=-4,
    vmax=4,
    stride=10,
    titles=titles,
    cmap="viridis",
    interpolation="nearest",
    dpi=150,
    bitrate=1800,
)

Rendering movie: 100%|██████████| 3001/3001 [06:46<00:00,  7.38it/s]


🎬 Movie saved: compare_movie.mp4


/ext3/miniforge3/lib/python3.12/site-packages/matplotlib/animation.py:908: UserWarning: Animation was deleted without rendering anything. This is most likely not intended. To prevent deletion, assign the Animation to a variable, e.g. `anim`, that exists until you output the Animation using `plt.show()` or `anim.save()`.
  warnings.warn(
